In [0]:
company_lookup = spark.table('plstocks.bronze_company_name_lookup')
company_lookup.display()

In [0]:
sector_lookup = spark.table('plstocks.silver_sector_lookup')
sector_lookup.display()

In [0]:
company_insights = company_lookup.join(sector_lookup, 'ticker', 'left').drop('insert_timestamp')
display(company_insights)

In [0]:
cashflow = spark.table('plstocks.silver_cashflow')
display(cashflow)

In [0]:
from pyspark.sql.functions import max, col

cashflow_agg = cashflow.groupBy('ticker').agg(max('date').alias('date'))
cashflow_result = cashflow.join(cashflow_agg, on=['ticker', 'date'])
cashflow_to_join = cashflow_result.select(col('ticker'), col('free_cashflow'))
display(cashflow_to_join)

In [0]:
company_insights = company_insights.join(cashflow_to_join, 'ticker', 'left').withColumnRenamed('free_cashflow', 'latest_free_cashflow')
display(company_insights)

In [0]:
stock_prices = spark.table('plstocks.silver_stocks_price')
display(stock_prices)

In [0]:
from pyspark.sql.functions import max, col

stocks_agg = stock_prices.groupBy('ticker').agg(max('date').alias('date'))
stocks_result = stock_prices.join(stocks_agg, on=['ticker', 'date'])
stocks_to_join = stocks_result.select(col('ticker'), col('close').cast('double'), col('ACTIVE'))
display(stocks_to_join)

In [0]:
company_insights = company_insights.join(stocks_to_join, 'ticker', 'left').withColumnRenamed('close', 'latest_stock_price').withColumnRenamed('ACTIVE', 'active_company')
display(company_insights)

In [0]:
financial_reports = spark.table('plstocks.silver_financial_reports')
display(financial_reports)

In [0]:
from pyspark.sql.functions import max, col

reports_agg = financial_reports.groupBy('ticker').agg(max('year').alias('year'))
reports_result = financial_reports.join(reports_agg, on=['ticker', 'year'])
reports_to_join = reports_result.select(col('ticker'), col('sales_revenue'))
display(reports_to_join)

In [0]:
company_insights = company_insights.join(reports_to_join, 'ticker', 'left')
display(company_insights)

In [0]:
financial_ratios = spark.table('plstocks.silver_financial_ratios')
display(financial_ratios)

In [0]:
from pyspark.sql.functions import max, col

ratios_agg = financial_ratios.groupBy('ticker').agg(max('insert_timestamp').alias('insert_timestamp'))
ratios_result = financial_ratios.join(ratios_agg, on=['ticker', 'insert_timestamp'])
display(ratios_result)
# reports_to_join = reports_result.select(col('ticker'), col('sales_revenue'))
# display(reports_to_join)